# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 08: Temporal Graph Networks (TGN) for Continuous-Time Graphs

### 1. Conceptual Intuition & Theoretical Background
Most graph models operate on static snapshots. In real-world systems (e.g. financial transactions, social networks), graphs evolve **continuously as a stream of timestamped interaction events**:
$$e_{ij}(t) = (u_i, u_j, t, \mathbf{e}_{ij})$$

**Temporal Graph Networks (TGN)** maintain continuous dynamic states using:
1. **Node Memory Bank ($s_i(t)$):** A state vector summarizing a node's entire historical interaction profile.
2. **Message Function:** Generates interaction messages when an event occurs:
$$m_i(t) = \text{msg}\left( s_i(t^-), s_j(t^-), \Delta t, \mathbf{e}_{ij} \right)$$
3. **Memory Updater:** Updates the memory using an RNN/GRU cell:
$$s_i(t) = \text{GRU}\left( \bar{m}_i(t), s_i(t^-) \right)$$
4. **Time Embedding (Bochner's Theorem):** Continuous time difference $\Delta t$ is mapped to harmonic Fourier features:
$$\Phi(\Delta t) = \sqrt{\frac{1}{d}} [\cos(\omega_1 \Delta t), \sin(\omega_1 \Delta t), \dots]$$

---

### 2. Scratch Python Implementation: Continuous Dynamic Graph Memory

In [ ]:
import numpy as np

class BochnerTimeEncoder:
    def __init__(self, time_dim=8):
        self.time_dim = time_dim
        # Fixed frequencies
        self.omegas = 1.0 / (10.0 ** np.linspace(0, 4, time_dim // 2))
        
    def encode(self, dt):
        angles = np.outer(dt, self.omegas)
        features = np.concatenate([np.cos(angles), np.sin(angles)], axis=-1)
        return features / np.sqrt(self.time_dim)

class TGNSimplifiedMemoryBank:
    def __init__(self, num_nodes, memory_dim=16):
        self.num_nodes = num_nodes
        self.mem_dim = memory_dim
        # Persistent memory state for each node
        self.memory = np.zeros((num_nodes, memory_dim))
        self.last_update_times = np.zeros(num_nodes)
        self.time_encoder = BochnerTimeEncoder(time_dim=8)
        
    def process_event(self, src, dst, timestamp):
        dt_src = timestamp - self.last_update_times[src]
        dt_dst = timestamp - self.last_update_times[dst]
        
        t_feat_src = self.time_encoder.encode(np.array([dt_src]))[0]
        
        # Interaction message
        src_msg = np.concatenate([self.memory[src][:8], self.memory[dst][:8]])
        
        # Update memory state using simple recurrent decay formula
        alpha = 0.5
        self.memory[src] = alpha * self.memory[src] + (1 - alpha) * np.pad(src_msg, (0, 0))[:self.mem_dim]
        self.memory[dst] = alpha * self.memory[dst] + (1 - alpha) * np.pad(src_msg, (0, 0))[:self.mem_dim]
        
        self.last_update_times[src] = timestamp
        self.last_update_times[dst] = timestamp

# Stream 5 Continuous Dynamic Interaction Events
bank = TGNSimplifiedMemoryBank(num_nodes=5, memory_dim=16)

events = [
    (0, 1, 10.5),
    (1, 2, 12.0),
    (0, 3, 15.2),
    (2, 4, 18.7),
    (3, 4, 25.1)
]

for src, dst, t in events:
    bank.process_event(src, dst, t)

print("--- Temporal Graph Network Dynamic State ---")
print(f"Node 0 Last Timestamp: {bank.last_update_times[0]:.1f} | Memory Norm: {np.linalg.norm(bank.memory[0]):.4f}")
print(f"Node 4 Last Timestamp: {bank.last_update_times[4]:.1f} | Memory Norm: {np.linalg.norm(bank.memory[4]):.4f}")
print("Dynamic memory bank successfully incorporates continuous interaction history!")

### 3. Step-by-Step Code Walkthrough
1. **Continuous Time Encoding**: Maps continuous temporal differences $\Delta t$ into bounded sinusoidal embeddings.
2. **Persistent State Bank**: Maintains rolling historical context across all nodes.
3. **Streaming Processing**: Updates states incrementally without re-running graph convolutions over the entire historical graph.

---

### 4. Industry Applications & Trade-offs
- **Real-Time Financial Fraud Detection:** Catching fraudulent transactions that occur within milliseconds across dynamic accounts.
- **Trade-off:** Memory staleness can occur if some nodes interact infrequently.

---

### 5. Top Interview Questions & Answers
**Q1: How do Temporal Graph Networks handle continuous time differences instead of discrete snapshot intervals?**
> **Answer:** TGNs use Bochner's theorem to project continuous time intervals $\Delta t$ into periodic Fourier feature vectors, allowing the network to capture both microsecond intervals and long-term trends.

**Q2: What is the "Memory Staleness" problem in TGNs?**
> **Answer:** If a node has not interacted for a long duration, its memory bank contains outdated states from its last historical event. TGNs resolve this by combining memory states with dynamic neighbor aggregation.